In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
===========================================================
PDF Header/Footer Cleaner + Page Deletion (Streamlit App)
===========================================================

Author   : Sanmathi
Year     : 2025
Version  : 1.0

Purpose  :
    This Streamlit application allows users to upload a PDF file,
    preview header/footer zones, remove unwanted header/footer
    content, and optionally delete selected pages. It provides
    a cleaned PDF for download.

Workflow :
    1. Upload a PDF file via Streamlit UI.
    2. Detect header/footer text using Docling.
    3. Preview header/footer zones visually with bounding boxes.
    4. Remove header/footer text, images, and drawings using PyMuPDF redactions.
    5. Optionally delete selected pages.
    6. Generate a cleaned PDF and preview results.
    7. Provide download button for the cleaned PDF.

Output   :
    - Cleaned PDF file with headers/footers removed.
    - Optionally excludes user-selected pages.
    - Visual previews of original and cleaned pages.

Dependencies:
    - streamlit
    - fitz (PyMuPDF)
    - tempfile
    - PIL (Pillow)
    - docling
    - re

Usage    :
    streamlit run pdf_cleaner.py

Notes    :
    - Header/footer detection uses Docling text conversion.
    - Redaction zones are applied to text, images, and drawings.
    - Final fallback ensures entire header/footer zones are cleared.
===========================================================
"""


import streamlit as st
import fitz  # PyMuPDF
import tempfile
from PIL import Image, ImageDraw
from docling.document_converter import DocumentConverter
import re

# ----------------- Docling detect header/footer text -----------------
def detect_header_footer_with_docling(pdf_path):
    try:
        converter = DocumentConverter()
        result = converter.convert(pdf_path)
        doc = result.document
        text = doc.export_to_text()
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        header = lines[0] if lines else ""
        footer = lines[-1] if lines else ""
        return header, footer
    except:
        return "", ""

# ----------------- Preview Zones -----------------
def preview_header_footer_zones(pdf_path, header_height, footer_height):
    doc = fitz.open(pdf_path)
    previews = []
    for page in doc:
        pix = page.get_pixmap(dpi=120)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        draw = ImageDraw.Draw(img)
        h_px = int(header_height * pix.height / page.rect.height)
        f_px = int(footer_height * pix.height / page.rect.height)
        draw.rectangle([0, 0, pix.width, h_px], outline="red", width=3)
        draw.rectangle([0, pix.height - f_px, pix.width, pix.height], outline="green", width=3)
        previews.append(img)
    return previews

# ----------------- Remove header/footer and delete pages -----------------
def remove_header_footer(input_path, output_path, header_height=50, footer_height=50, delete_pages=[]):
    pdf = fitz.open(input_path)
    detected_header, detected_footer = detect_header_footer_with_docling(input_path)
    new_pdf = fitz.open()

    for i, page in enumerate(pdf):
        if (i + 1) in delete_pages:
            continue

        rect = page.rect
        header_zone = fitz.Rect(0, 0, rect.width, header_height)
        footer_zone = fitz.Rect(0, rect.height - footer_height, rect.width, rect.height)

        # Redact text blocks
        blocks = page.get_text("dict")["blocks"]
        for b in blocks:
            if "lines" not in b:
                continue
            (x0, y0, x1, y1) = b["bbox"]
            text = ""
            for l in b["lines"]:
                for s in l["spans"]:
                    text += s["text"].strip() + " "
            text = text.strip()
            is_page_number = bool(re.search(r"Page\s*\d+", text))
            match_docling = (text == detected_header or text == detected_footer) and not is_page_number
            match_zone = (y1 < header_height or y0 > rect.height - footer_height) and not is_page_number
            if match_docling or match_zone:
                page.add_redact_annot(fitz.Rect(x0, y0, x1, y1), fill=(1, 1, 1))

        # Redact images
        for img in page.get_images(full=True):
            xref = img[0]
            try:
                for r in page.get_image_bbox(xref):
                    if header_zone.intersects(r) or footer_zone.intersects(r):
                        page.add_redact_annot(r, fill=(1, 1, 1))
            except ValueError:
                continue

        # Redact drawings
        drawings = page.get_drawings()
        for d in drawings:
            r = d["rect"]
            if header_zone.intersects(r) or footer_zone.intersects(r):
                page.add_redact_annot(r, fill=(1, 1, 1))

        # 🔥 Final fallback: forcefully redact entire header/footer zones
        page.add_redact_annot(header_zone, fill=(1, 1, 1))
        page.add_redact_annot(footer_zone, fill=(1, 1, 1))

        page.apply_redactions()
        new_pdf.insert_pdf(pdf, from_page=i, to_page=i)

    new_pdf.save(output_path)



# ----------------- Streamlit UI -----------------
st.title("🧹 PDF Header/Footer Cleaner + Manual Page Deletion")

uploaded_file = st.file_uploader("Upload PDF", type=["pdf"])
header_height = st.slider("Header height (points)", 0, 150, 100)
footer_height = st.slider("Footer height (points)", 0, 150, 100)

if uploaded_file:
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_input:
        temp_input.write(uploaded_file.read())
        temp_input_path = temp_input.name

    doc = fitz.open(temp_input_path)
    total_pages = len(doc)

    st.subheader("📸 Original Header/Footer Zones")
    previews = preview_header_footer_zones(temp_input_path, header_height, footer_height)
    for i, img in enumerate(previews):
        st.image(img, caption=f"Original Page {i+1}", use_column_width=True)

    delete_pages = st.multiselect(
        "🗑️ Select pages to delete (1-based index)",
        options=list(range(1, total_pages + 1)),
        help="Choose pages you want to exclude from the final PDF"
    )

    if st.button("🧼 Clean and Preview"):
        temp_output_path = temp_input_path.replace(".pdf", "_cleaned.pdf")
        remove_header_footer(temp_input_path, temp_output_path, header_height, footer_height, delete_pages)

        st.subheader("🧼 Cleaned PDF Preview")
        cleaned_doc = fitz.open(temp_output_path)
        for i, page in enumerate(cleaned_doc):
            pix = page.get_pixmap(dpi=120)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            st.image(img, caption=f"Cleaned Page {i+1}", use_column_width=True)

        st.success("✅ Header/Footer removed & selected pages deleted!")

        with open(temp_output_path, "rb") as f:
            st.download_button("📥 Download Cleaned PDF", f, file_name="cleaned.pdf")










